In [26]:
import pandas as pd

## Создай датафрейм с названием views с двумя столбцами datetime и user, прочитав файл feed-views.log.
- Приведи столбец datetime к типу datetime64[ns] (dtype).
- Извлеки из datetime год, месяц, день, час, минуту и секунду в новые столбцы.

In [27]:
views = pd.read_csv('../data/feed-views.log', sep='\t', names=['datetime', 'user'])
views.head(1)

,datetime,user
0,2020-04-17 12:01:08.463179,artem


In [28]:
print(views.dtypes)

datetime    str
user        str
dtype: object


In [29]:
views['datetime'] = views['datetime'].astype('datetime64[ns]')
print(views.dtypes)

datetime    datetime64[ns]
user                   str
dtype: object


In [30]:
views['year'] = views['datetime'].dt.year
views['month'] = views['datetime'].dt.month
views['day'] = views['datetime'].dt.day
views['hour'] = views['datetime'].dt.hour
views['minute'] = views['datetime'].dt.minute
views['second'] = views['datetime'].dt.second

views.head(1)

,datetime,user,year,month,day,hour,minute,second
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8


## Создай новый столбец daytime.
- Присваивай значение времени суток, если час попадает в заданный интервал (например, «afternoon», если час > 11 и ≤ 17).
  - 0-3:59 = night,
  - 4-6:59 = early morning,
  - 7-10:59 = morning,
  - 11-16:59 = afternoon,
  - 17-19:59 = early evening,
  - 20-23:59 = evening.
- Для решения подзадачи используй метод cut.
- Назначь столбец user индексом.

In [31]:
bins = [0, 4, 7, 11, 17, 20, 24]
labels = ['night', 'early morning', 'morning', 'afternoon', 'early evening', 'evening']

views['daytime'] = pd.cut(views['hour'], bins=bins, labels=labels, right=False)
views.head(1)

,datetime,user,year,month,day,hour,minute,second,daytime
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8,afternoon


In [32]:
views.set_index('user', inplace=True)
views.head(1)

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon


## Посчитай количество элементов в датафрейме.
- Используй метод count().
- Посчитай число элементов в каждой категории времени суток с помощью value_counts().

In [33]:
views.count()

datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [34]:
views['daytime'].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

## Отсортируй значения в датафрейме одновременно по часам, минутам и секундам по возрастанию (не по одному столбцу по очереди).

In [35]:
views = views.sort_values(by=['hour', 'minute', 'second'])
views.head()

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night


## Вычисли минимум и максимум для часов и моду для категорий daytime.
- Найди максимальный час для строк, где время суток - night.
- Найди минимальный час для строк, где время суток - morning.
- Дополнительно узнай, кто посещал страницу в эти часы, и приведи один пример.
- Вычисли моду для часа и для daytime.

In [36]:
print(f'минимум для hour: {views['hour'].min()}')
print(f'максимум для hour: {views['hour'].max()}')
print(f'мода для daytime: {views['daytime'].mode()[0]}')

минимум для hour: 0
максимум для hour: 23
мода для daytime: evening


In [37]:
max_night_hour = views[views['daytime'] == 'night']['hour'].max()
min_morning_hour = views[views['daytime'] == 'morning']['hour'].min()

print(f'максимум hour, где daytime == night: {max_night_hour}')
print(f'минимум hour, где daytime == morning: {min_morning_hour}')

максимум hour, где daytime == night: 3
минимум hour, где daytime == morning: 8


In [38]:
print(f'кто посещал страницу в максимальный час ночи: {views[views['hour'] == max_night_hour].index[0]}')
print(f'кто посещал страницу в минимальный час утра: {views[views['hour'] == min_morning_hour].index[0]}')

кто посещал страницу в максимальный час ночи: konstantin
кто посещал страницу в минимальный час утра: alexander


In [39]:
print(f'мода для hour: {views['hour'].mode()[0]}')
print(f'мода для daytime: {views['daytime'].mode()[0]}')

мода для hour: 22
мода для daytime: evening


## Покажи три самых ранних и три самых поздних часа дня и соответствующие имена пользователей, используя nsmalles() и nlargest().

In [40]:
print(f'3 самых ранних часа:')
views.nsmallest(3, 'hour')[['hour']]

3 самых ранних часа:


,hour
user,
valentina,0
valentina,0
pavel,0


In [41]:
print(f'3 самых поздних часа:')
views.nlargest(3, 'hour')[['hour']]

3 самых поздних часа:


,hour
user,
ekaterina,23
ekaterina,23
ekaterina,23


## Используй метод describe() для получения базовой статистики по столбцам.
- Чтобы найти самый популярный интервал посещений, вычисли интерквартильный размах для часа: извлеки нужные значения из результата describe() и сохрани их в переменную iqr.

In [42]:
views.describe()

,datetime,year,month,day,hour,minute,second
count,1076,1076.0,1076.000000,1076.000000,1076.000000,1076.000000,1076.000000
mean,2020-05-10 09:00:41.211420672,2020.0,4.870818,13.552974,16.249071,29.629182,29.500929
min,2020-04-17 12:01:08.463179,2020.0,4.000000,1.000000,0.000000,0.000000,0.000000
25%,2020-05-10 01:13:49.857472,2020.0,5.000000,11.000000,13.000000,14.000000,14.000000
50%,2020-05-11 22:48:35.302552832,2020.0,5.000000,13.000000,19.000000,29.000000,30.000000
75%,2020-05-14 14:44:34.749530624,2020.0,5.000000,15.000000,22.000000,46.000000,45.000000
max,2020-05-22 10:36:14.662600,2020.0,5.000000,30.000000,23.000000,59.000000,59.000000
std,NaN,0.0,0.335557,4.906567,6.955490,17.689388,17.405506


In [43]:
desc = views.describe()

q1 = desc.loc['25%', 'hour']
q3 = desc.loc['75%', 'hour']
iqr = q3 - q1

print(f'iqr: {iqr}')

iqr: 9.0


In [45]:
views.info()

<class 'pandas.DataFrame'>
Index: 1076 entries, valentina to alexander
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  1076 non-null   datetime64[ns]
 1   year      1076 non-null   int32         
 2   month     1076 non-null   int32         
 3   day       1076 non-null   int32         
 4   hour      1076 non-null   int32         
 5   minute    1076 non-null   int32         
 6   second    1076 non-null   int32         
 7   daytime   1076 non-null   category      
dtypes: category(1), datetime64[ns](1), int32(6)
memory usage: 75.6+ KB


In [46]:
views.count()

datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [21]:
views['daytime'].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

In [47]:
views.loc[views.daytime == 'night'].hour.idxmax()

'konstantin'

In [48]:
views.loc[views.daytime == 'morning'].hour.idxmin()

'alexander'

In [49]:
views.hour.mode()

0    22
Name: hour, dtype: int32

In [50]:
views.daytime.mode()

0    evening
Name: daytime, dtype: category
Categories (6, str): ['night' < 'early morning' < 'morning' < 'afternoon' < 'early evening' < 'evening']